## Test case generation with Large Language Models

In this series of exercises, we will investigate the use of LLM to generate test cases.

### Step 1: Our reference code

As opposed to the previous experience with code generation - where we had valid test cases - we assume here that we have valid code for given software requirements. Our task now is to generate test cases for valid code.



In [1]:
#the same code is saved in the python script function_01.py

original_function="""def racer_disqualified(times, winner_times, n_penalties, penalties):
    \"""
    Determines if a racer is disqualified based on their times, penalties, and winner times.

    Parameters:
        times (list of int): List of the racer's times for three events.
        winner_times (list of int): List of winner times for the same three events.
        n_penalties (int): Number of penalties the racer incurred.
        penalties (list of int): List of penalty values.

    Returns:
        bool: True if the racer is disqualified, False otherwise.

    Raises:
        ValueError: If inputs do not meet the required types or constraints.
    \"""
    # Input validation
    if not (isinstance(times, list) and len(times) == 3 and all(isinstance(t, int) for t in times)):
        raise ValueError("times must be a list of three integers.")

    if not (isinstance(winner_times, list) and len(winner_times) == 3 and all(isinstance(wt, int) for wt in winner_times)):
        raise ValueError("winner_times must be a list of three integers.")

    if not isinstance(n_penalties, int):
        raise ValueError("n_penalties must be an integer.")

    if not (isinstance(penalties, list) and all(isinstance(p, int) for p in penalties)):
        raise ValueError("penalties must be a list of integers.")

    if n_penalties != len(penalties):
        raise ValueError("n_penalties must match the length of the penalties list.")

    disqualified = False
    tot_penalties = 0

    # Calculate total penalties and check for any excessive penalty
    for penalty in penalties:
        tot_penalties += penalty
        if penalty > 100:
            disqualified = True

    # Check for disqualification based on total penalties or number of penalties
    if tot_penalties > 100 or n_penalties > 5:
        disqualified = True

    # Check if any time exceeds 1.5 times the corresponding winner time
    for i in range(3):
        max_time = winner_times[i] * 1.5
        if times[i] > max_time:
            disqualified = True

    return disqualified"""


file_path = "./function_01.py"

with open(file_path, 'w') as file:
    file.write(original_function + "\n")



def racer_disqualified(times, winner_times, n_penalties, penalties):
    """
    Determines if a racer is disqualified based on their times, penalties, and winner times.

    Parameters:
        times (list of int): List of the racer's times for three events.
        winner_times (list of int): List of winner times for the same three events.
        n_penalties (int): Number of penalties the racer incurred.
        penalties (list of int): List of penalty values.

    Returns:
        bool: True if the racer is disqualified, False otherwise.

    Raises:
        ValueError: If inputs do not meet the required types or constraints.
    """
    # Input validation
    if not (isinstance(times, list) and len(times) == 3 and all(isinstance(t, int) for t in times)):
        raise ValueError("times must be a list of three integers.")

    if not (isinstance(winner_times, list) and len(winner_times) == 3 and all(isinstance(wt, int) for wt in winner_times)):
        raise ValueError("winner_times must be a list of three integers.")

    if not isinstance(n_penalties, int):
        raise ValueError("n_penalties must be an integer.")

    if not (isinstance(penalties, list) and all(isinstance(p, int) for p in penalties)):
        raise ValueError("penalties must be a list of integers.")

    if n_penalties != len(penalties):
        raise ValueError("n_penalties must match the length of the penalties list.")

    disqualified = False
    tot_penalties = 0

    # Calculate total penalties and check for any excessive penalty
    for penalty in penalties:
        tot_penalties += penalty
        if penalty > 100:
            disqualified = True

    # Check for disqualification based on total penalties or number of penalties
    if tot_penalties > 100 or n_penalties > 5:
        disqualified = True

    # Check if any time exceeds 1.5 times the corresponding winner time
    for i in range(3):
        max_time = winner_times[i] * 1.5
        if times[i] > max_time:
            disqualified = True

    return disqualified

### Step 2: Define some pytest test cases

We now setup an environment to run test cases and obtain the coverage of test cases. To start, we define a couple of test cases with the PyTest library.

In [2]:
import sys
from pathlib import Path
import pytest
import ipytest

# Use the laboratory folder as the notebook working directory.
LAB_DIR = Path.cwd()
assert (LAB_DIR / "function_01.py").is_file(), "Run this notebook from LABS/lab10"
sys.path.insert(0, str(LAB_DIR.parent))

# The manual baseline tests exercise each rule and its strict boundary.
manual_test_source = 'import pytest\nfrom lab10.function_01 import racer_disqualified\n\n@pytest.mark.parametrize("times, penalties, expected", [\n    ([100, 100, 100], [], False),\n    ([150, 150, 150], [], False),\n    ([151, 100, 100], [], True),\n    ([100, 151, 100], [], True),\n    ([100, 100, 151], [], True),\n    ([100, 100, 100], [100], False),\n    ([100, 100, 100], [101], True),\n    ([100, 100, 100], [50, 50], False),\n    ([100, 100, 100], [50, 51], True),\n    ([100, 100, 100], [10] * 5, False),\n    ([100, 100, 100], [10] * 6, True),\n])\ndef test_disqualification_rules(times, penalties, expected):\n    assert racer_disqualified(times, [100, 100, 100], len(penalties), penalties) is expected\n\n@pytest.mark.parametrize("args", [\n    ("100,100,100", [100, 100, 100], 0, []),\n    ([100, 100], [100, 100, 100], 0, []),\n    ([100, 100, 1.5], [100, 100, 100], 0, []),\n    ([100, 100, 100], "100,100,100", 0, []),\n    ([100, 100, 100], [100, 100], 0, []),\n    ([100, 100, 100], [100, 100, "100"], 0, []),\n    ([100, 100, 100], [100, 100, 100], "0", []),\n    ([100, 100, 100], [100, 100, 100], 0, ""),\n    ([100, 100, 100], [100, 100, 100], 1, [1.5]),\n    ([100, 100, 100], [100, 100, 100], 2, [10]),\n])\ndef test_invalid_inputs(args):\n    with pytest.raises(ValueError):\n        racer_disqualified(*args)\n'
(LAB_DIR / "test_function_01_manual.py").write_text(manual_test_source)
exec(compile(manual_test_source, "test_function_01_manual.py", "exec"), globals())
ipytest.autoconfig(raise_on_error=True)

def run_tests():
    return ipytest.run("-vv")

run_tests()

======================================= test session starts ========================================
platform darwin -- Python 3.11.16, pytest-9.1.1, pluggy-1.7.0 -- /private/tmp/llms-lab10-venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/nicolabeccaceci/Desktop/LLMs/LABS/lab10
plugins: cov-7.1.0, platformdirs-4.13.5
collecting ... 

collected 21 items

t_1032c35a5d7f4ed4b89f850c009b245f.py::test_disqualification_rules[times0-penalties0-False] 

PASSED [  4%]


t_1032c35a5d7f4ed4b89f850c009b245f.py::test_disqualification_rules[times1-penalties1-False] 

PASSED [  9%]

...

PASSED                     [ 95%]


t_1032c35a5d7f4ed4b89f850c009b245f.py::test_invalid_inputs[args9] 

PASSED                     [100%]



======================================== 21 passed in 0.10s ========================================


<ExitCode.OK: 0>

### Step 3: Computing the pass rate

The first objective of our analysis is computing the pass rate of the test cases.

The pass rate for a test suite is defined as the ratio between the passing test cases and all the test cases executed.

Notice that this ratio is computed in the same way as the Functional Correctness when you are comparing generated code against an existing test suite, but there is a subtle difference in what we are measuring: 
- when we compute **functional correctness**, we have a correct test suite, and we are verifying if the code complies to requirements by executing the test cases.
- when we compute the **pass rate**, we have correct code, and we are verifying if the test cases comply to the requirements by executing them against the code.

For now, we are defining the test cases manually: we make sure that the pass rate is 100%.

In [3]:
import os
import re
import subprocess
import sys
import xml.etree.ElementTree as ET
from pathlib import Path

ARTIFACTS = LAB_DIR / "execution"
ARTIFACTS.mkdir(exist_ok=True)
RUN_ENV = dict(os.environ)
RUN_ENV["PYTHONPATH"] = str(LAB_DIR.parent) + os.pathsep + RUN_ENV.get("PYTHONPATH", "")
RUN_ENV["PYTHONDONTWRITEBYTECODE"] = "1"


def run_command(arguments, tag):
    result = subprocess.run(
        [sys.executable, "-B", *arguments], cwd=LAB_DIR, env=RUN_ENV,
        capture_output=True, text=True, timeout=120,
    )
    (ARTIFACTS / f"{tag}.log").write_text(result.stdout + result.stderr)
    return result


def parse_test_results(xml_path):
    root = ET.parse(xml_path).getroot()
    cases = root.findall(".//testcase")
    if not cases:
        raise RuntimeError(f"No tests collected; see {xml_path}")
    outcomes = {}
    for case in cases:
        key = case.get("classname", "") + "::" + case.get("name", "")
        outcomes[key] = (
            "error" if case.find("error") is not None else
            "failed" if case.find("failure") is not None else
            "skipped" if case.find("skipped") is not None else "passed"
        )
    counts = {state: sum(value == state for value in outcomes.values())
              for state in ("passed", "failed", "error", "skipped")}
    counts["total"] = len(cases)
    counts["pass_rate"] = counts["passed"] / counts["total"]
    counts["outcomes"] = outcomes
    return counts


def run_suite(test_file, tag):
    xml_path = ARTIFACTS / f"{tag}.xml"
    result = run_command([
        "-m", "pytest", str(test_file), "-q", "--tb=short", "--disable-warnings",
        f"--junitxml={xml_path}",
    ], tag)
    if result.returncode not in (0, 1):
        raise RuntimeError(f"pytest infrastructure error: {tag}; see its log")
    counts = parse_test_results(xml_path)
    if counts["error"]:
        raise RuntimeError(f"Collection/setup error in {tag}; see its log")
    return result, counts


result, manual_baseline = run_suite("test_function_01_manual.py", "manual-baseline")
errors, failures, passes = (manual_baseline[key] for key in ("error", "failed", "passed"))
print(f"# Passed: {passes}")
print(f"# Failed: {failures}")
print(f"# Errors: {errors}")
pass_rate = manual_baseline["pass_rate"]
print(f"Pass Rate: {pass_rate:.2%}")
assert pass_rate == 1.0

# Passed: 21
# Failed: 0
# Errors: 0
Pass Rate: 100.00%


### Step 4: Compute the coverage

To compute the coverage of a test suite over a function or a set of functions, we can use the coverage library.

pip install pytest-cov

Once we have the coverage module installed, it is possible to launch the coverage by launching the following command line instructions:
- coverage run -m pytest test_function_name
- coverage report -m

In this code section, define multiple subprocess runs to obtain the results of the coverage computation inside a variable.

In [4]:
def measure_coverage(test_file, tag):
    data_file = ARTIFACTS / f".coverage-{tag}"
    xml_path = ARTIFACTS / f"{tag}-coverage-tests.xml"
    result = run_command([
        "-m", "coverage", "run", f"--data-file={data_file}",
        "--source=lab10.function_01", "-m", "pytest", str(test_file),
        "-q", "--tb=short", f"--junitxml={xml_path}",
    ], f"{tag}-coverage-run")
    if result.returncode not in (0, 1):
        raise RuntimeError(f"Coverage test execution failed: {tag}")
    result2 = run_command([
        "-m", "coverage", "report", f"--data-file={data_file}",
        "--include=*/lab10/function_01.py", "-m",
    ], f"{tag}-coverage-report")
    if result2.returncode != 0:
        raise RuntimeError(result2.stdout + result2.stderr)
    line = next(line for line in result2.stdout.splitlines() if "function_01.py" in line)
    coverage = float(re.search(r"(\d+(?:\.\d+)?)%", line).group(1))
    return result, result2, coverage


result, result2, coverage = measure_coverage("test_function_01_manual.py", "manual")
print(result2.stdout)
print(f"Coverage: {coverage}%")

Name             Stmts   Miss  Cover   Missing
----------------------------------------------
function_01.py      24      0   100%
----------------------------------------------
TOTAL               24      0   100%

Coverage: 100.0%


### Step 5: Introducing mutations

To try out mutation testing, produce a set of variants of the function by changing operators and values. Save all these variants in a dictionary of mutations like in the example below.

Remeber to introduce a single mutant in each mutated version of the function.

**Note**: several tools exist to automate mutation. You can refer to the libraries mutatest and mutpy to generate automatic mutations for test cases written with pytest. In this example, we will introduce mutations manually.

In [5]:
# Each mutant changes exactly one expression in the reference implementation.
def mutate_once(old, new):
    assert original_function.count(old) == 1
    mutated = original_function.replace(old, new, 1)
    compile(mutated, "function_01.py", "exec")
    return mutated

mutant1 = mutate_once("if penalty > 100:", "if penalty < 100:")
mutant2 = mutate_once("tot_penalties > 100", "tot_penalties >= 100")
mutant3 = mutate_once("n_penalties > 5", "n_penalties >= 5")
mutant4 = mutate_once("times[i] > max_time", "times[i] >= max_time")
mutant5 = mutate_once("return disqualified", "return True")
mutants = [mutant1, mutant2, mutant3, mutant4, mutant5]
mutant_descriptions = [
    "reverse individual penalty comparison", "include total penalty boundary",
    "include penalty-count boundary", "include time boundary", "always disqualify",
]
print(f"Prepared {len(mutants)} single-change mutants.")

Prepared 5 single-change mutants.


### Step 6: Calculating Mutation Score

Now cycle over the list of mutants. For every mutant, overwrite the function function_01.py and re-execute the test cases. For each mutant you can compute the following outcome:
- Mutant killed: one or more test cases failed
- Mutant survived: all test cases passed

At the end of the iteration over mutants, compute the mutation score:
- Mutation score = killed mutants / total number of mutants. The surviving fraction is reported separately.

In [6]:
file_path = LAB_DIR / "function_01.py"


def write_reference(source):
    file_path.write_text(source + "\n")
    # A same-sized mutation made within one timestamp tick can otherwise reuse .pyc.
    for cache in (LAB_DIR / "__pycache__").glob("function_01.*.pyc"):
        cache.unlink()


def evaluate_mutants(test_file, baseline, tag):
    passing_baseline = {key for key, value in baseline["outcomes"].items() if value == "passed"}
    details = []
    try:
        for index, (mutant, description) in enumerate(zip(mutants, mutant_descriptions), 1):
            write_reference(mutant)
            _, counts = run_suite(test_file, f"{tag}-mutant-{index}")
            # An already-failing generated test cannot provide evidence of killing a mutant.
            newly_failing = sorted(key for key, state in counts["outcomes"].items()
                                   if key in passing_baseline and state == "failed")
            killed = bool(newly_failing)
            details.append({"mutant": index, "change": description,
                            "killed": killed, "newly_failing_tests": newly_failing})
            print(f"Mutant {index}: {'killed' if killed else 'survived'} ({description})")
    finally:
        write_reference(original_function)
    return details


manual_mutations = evaluate_mutants("test_function_01_manual.py", manual_baseline, "manual")
killed_mutants = sum(entry["killed"] for entry in manual_mutations)
survived_mutants = len(mutants) - killed_mutants
mutation_score = killed_mutants / len(mutants)
print(f"Killed: {killed_mutants}; survived: {survived_mutants}")
print(f"Mutation score: {mutation_score:.2%}")

Mutant 1: killed (reverse individual penalty comparison)


Mutant 2: killed (include total penalty boundary)


Mutant 3: killed (include penalty-count boundary)


Mutant 4: killed (include time boundary)


Mutant 5: killed (always disqualify)
Killed: 5; survived: 0
Mutation score: 100.00%


### Step 7 : Generating tests with LLMs

We will consider again at least two alternatives for test case generation:
- a model from HuggingFace, e.g., CodeLLAMA
- a chat engine, e.g., ChatGPT or Qwen2.5

With each engine, we will generate a new test file (e.g., test_function_01_gpt.py, and test_function_01_llama.py), and replicate the pass rate, coverage and mutation analysis performed before with pre-defined test cases.

The repository already supplies GPT, Copilot and Qwen test examples. The following comparison evaluates those files as supplied examples; it does not claim to regenerate their outputs or independently verify their original model provenance. A new Codex-authored suite is included as an explicitly labeled fresh generation. The supplied Copilot example calls `pytest.main()` at import time; an execution copy removes only that top-level runner call so pytest can collect it normally. Original supplied test files remain unchanged. Generated tests with incorrect expectations are kept and reported as failures; mutation kills count only failures introduced among tests that passed on the reference.

In [7]:
import ast
import datetime
import hashlib
import json

# Fresh generation by the Codex agent handling this lab; no external API is called.
codex_test_source = '"""Fresh Codex-authored tests generated for this laboratory execution."""\nimport pytest\nfrom lab10.function_01 import racer_disqualified\n\n@pytest.mark.parametrize("event", range(3))\n@pytest.mark.parametrize("offset, expected", [(-1, False), (0, False), (1, True)])\ndef test_time_limit_for_each_event(event, offset, expected):\n    winner = [100, 200, 300]\n    times = winner.copy()\n    times[event] = int(1.5 * winner[event]) + offset\n    assert racer_disqualified(times, winner, 0, []) is expected\n\n@pytest.mark.parametrize("penalties, expected", [\n    ([], False), ([99], False), ([100], False), ([101], True),\n    ([50, 50], False), ([50, 51], True), ([1] * 5, False), ([1] * 6, True),\n])\ndef test_penalty_limits(penalties, expected):\n    assert racer_disqualified([10, 20, 30], [10, 20, 30], len(penalties), penalties) is expected\n\n@pytest.mark.parametrize("position, invalid", [\n    (0, (10, 20, 30)), (0, [10, 20]), (0, [10, 20, "30"]),\n    (1, None), (1, [10, 20]), (1, [10, 20, 30.5]),\n    (2, "0"), (3, ()), (3, ["10"]), (3, [10]),\n])\ndef test_reference_input_contract(position, invalid):\n    args = [[10, 20, 30], [10, 20, 30], 0, []]\n    args[position] = invalid\n    with pytest.raises(ValueError):\n        racer_disqualified(*args)\n'
(LAB_DIR / "test_function_01_codex.py").write_text(codex_test_source)

# Evaluate the supplied examples alongside the new, explicitly labeled suite.
test_files = [
    "test_function_01_gpt.py", "test_function_01_copilot.py",
    "test_function_01_qwen.py", "test_function_01_codex.py",
]
suite_dir = ARTIFACTS / "suites"
suite_dir.mkdir(exist_ok=True)
comparison = {}
provenance = []

for test_file in test_files:
    source_path = LAB_DIR / test_file
    test_source = source_path.read_text()
    tree = ast.parse(test_source)
    runner_calls = [node for node in tree.body if isinstance(node, ast.Expr)
                    and isinstance(node.value, ast.Call)
                    and isinstance(node.value.func, ast.Attribute)
                    and isinstance(node.value.func.value, ast.Name)
                    and node.value.func.value.id == "pytest"
                    and node.value.func.attr == "main"]
    # Remove only side-effectful top-level test-runner invocations in execution copies.
    lines = test_source.splitlines(keepends=True)
    for node in reversed(runner_calls):
        del lines[node.lineno - 1:node.end_lineno]
    execution_path = suite_dir / test_file
    execution_path.write_text("".join(lines))
    provenance.append({
        "file": test_file,
        "origin": "fresh Codex generation" if "codex" in test_file else "supplied repository example",
        "source_sha256": hashlib.sha256(test_source.encode()).hexdigest(),
        "removed_top_level_pytest_main_calls": len(runner_calls),
    })

    print("\nDoing:", test_file)
    write_reference(original_function)
    tag = source_path.stem
    _, baseline = run_suite(execution_path, tag + "-baseline")
    number_of_tests = baseline["total"]
    pass_rate = baseline["pass_rate"]
    print(f"Number of tests: {number_of_tests}")
    print(f"Pass Rate: {pass_rate:.2%} ({baseline['failed']} failed)")
    if baseline["failed"]:
        print("Failing reference tests:", [key for key, state in baseline["outcomes"].items() if state == "failed"])
    _, coverage_report, coverage = measure_coverage(execution_path, tag)
    print(f"Coverage: {coverage}%")
    mutation_details = evaluate_mutants(execution_path, baseline, tag)
    killed_mutants = sum(entry["killed"] for entry in mutation_details)
    survived_mutants = len(mutants) - killed_mutants
    mutation_score = killed_mutants / len(mutants)
    print(f"Mutation score: {mutation_score:.2%}")
    comparison[test_file] = {
        "tests": number_of_tests, "passed": baseline["passed"], "failed": baseline["failed"],
        "pass_rate": pass_rate, "statement_coverage_percent": coverage,
        "killed_mutants": killed_mutants, "survived_mutants": survived_mutants,
        "mutation_score": mutation_score, "mutants": mutation_details,
    }

write_reference(original_function)
(ARTIFACTS / "comparison.json").write_text(json.dumps(comparison, indent=2) + "\n")
(ARTIFACTS / "test-provenance.json").write_text(json.dumps(provenance, indent=2) + "\n")
print("\nOriginal function restored:", file_path.read_text() == original_function + "\n")


Doing: test_function_01_gpt.py


Number of tests: 12
Pass Rate: 91.67% (1 failed)
Failing reference tests: ['execution.suites.test_function_01_gpt::test_disqualified_due_to_exceeding_time_limit']


Coverage: 96.0%


Mutant 1: killed (reverse individual penalty comparison)


Mutant 2: survived (include total penalty boundary)


...

Mutant 1: killed (reverse individual penalty comparison)


Mutant 2: killed (include total penalty boundary)


Mutant 3: killed (include penalty-count boundary)


Mutant 4: killed (include time boundary)


Mutant 5: killed (always disqualify)
Mutation score: 100.00%

Original function restored: True
